# Lab: from raw vibrations to honest error bars

**Why this lab exists.** For the Z24 bridge (your second structure), you will probably receive *raw accelerations*, not ready-made natural frequencies. This lab teaches you, step by step, how to turn vibrations into frequencies, and connects that to your KW51 project. You work on a simulated 3-mass "bridge" where the true answers are known, so you can check yourself.

**How to use it.**
- Run cells top to bottom.
- Cells marked `# YOUR CODE` raise `NotImplementedError` until you write them.
- After each exercise, a `check_labN(...)` cell prints **PASS** or a hint.
- Answer the 💬 questions in your lab notebook *in your own words*; they are what judges and mentors will ask.
- Don't read `check_*` in `labkit.py` before you try: it would give the answers away.

| Lab | Skill | Time |
|---|---|---|
| 1 | Natural frequencies from mass and stiffness (eigenvalues) | 30 min |
| 2 | Frequencies from noisy ambient vibration (spectrum + peak picking) | 1 h |
| 3 | Stochastic subspace identification and stabilization diagrams | 1–2 h |
| 4 | Temperature and freezing: fit the physics law | 1 h |
| 5 | Conformal prediction: random vs seasonal split | 1 h |


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import linalg, signal, optimize
import labkit as L
%matplotlib inline

## Lab 1: natural frequencies from M and K
A structure with mass matrix **M** and stiffness matrix **K** vibrates freely in patterns (modes) satisfying **K φ = ω² M φ**. That is a *generalized eigenvalue problem*. Each eigenvalue λ = ω² gives a natural frequency **f = √λ / (2π)** in Hz.

**Exercise 1.** Compute the three natural frequencies of `L.bridge_model()`.
Hint: `linalg.eigh(K, M, eigvals_only=True)` returns the eigenvalues λ.

In [ ]:
M, K = L.bridge_model()
# YOUR CODE: freqs = ...
raise NotImplementedError("Lab 1: compute freqs (Hz)")

In [ ]:
L.check_lab1(freqs); freqs

💬 **Q1.** Make the springs 1 % stiffer (`L.bridge_model(1.01)`). By how many percent does each frequency change? Explain why from f ∝ √k. *(This is exactly the physics in your KW51 model.)*

## Lab 2: frequencies from ambient vibration
Real bridges are shaken by wind and traffic: random forces we don't measure. `L.simulate_ambient` returns 15 minutes of noisy acceleration at the 3 masses, sampled at 50 Hz.

The **power spectral density (PSD)** shows how much vibration energy sits at each frequency. Natural frequencies appear as **peaks**.

**Exercise 2.**
1. Compute the PSD of each channel with `signal.welch(y, fs, nperseg=...)` (use `axis=-1`), then sum over channels.
2. Find the 3 largest peaks with `signal.find_peaks` (try the `prominence=` argument).
3. Store their frequencies in `est` (Hz).

In [ ]:
y, fs = L.simulate_ambient(*L.bridge_model())
t = np.arange(y.shape[1]) / fs
plt.figure(figsize=(8, 2)); plt.plot(t[:1500], y[0, :1500]); plt.xlabel("time [s]"); plt.ylabel("acc [m/s²]"); plt.title("Looks like noise..."); plt.show()

In [ ]:
# YOUR CODE: f_psd, P = signal.welch(...);  P_sum = ...;  peaks = signal.find_peaks(...)[0];  est = ...
raise NotImplementedError("Lab 2: spectrum + peak picking")

In [ ]:
plt.figure(figsize=(8, 3)); plt.semilogy(f_psd, P_sum); [plt.axvline(e, color="r", ls="--") for e in est]
plt.xlim(0, 8); plt.xlabel("Frequency [Hz]"); plt.ylabel("PSD"); plt.show()
L.check_lab2(est); est

💬 **Q2.** Try `nperseg` = 256, 1024, 4096. What happens to the peaks and the frequency resolution (`fs/nperseg`)? What's the trade-off?
💬 **Q3.** Peak picking gives no reliable damping. Why might damping matter for monitoring?

## Lab 3: stochastic subspace identification (SSI)
SSI fits a small linear state-space model to the vibration data. It's the method the KW51 authors used to produce the frequencies you've been analysing. `L.ssi_cov` is provided; read its docstring for the 4 ideas behind it.

Because we don't know the right model *order*, we fit many orders. **Physical modes stay at the same frequency for every order** and form vertical columns in a *stabilization diagram*. Spurious "noise modes" jump around.

**Exercise 3.**
1. Run SSI and draw the stabilization diagram.
2. Decide which 3 columns are physical.
3. For each, average the frequency and damping over the orders where it appears (e.g. all poles within ±1 % of the column centre).
4. Store `f_ssi` (3 values, Hz) and `z_ssi` (3 damping ratios, e.g. 0.017 for 1.7 %).

In [ ]:
res = L.ssi_cov(y, fs)
plt.figure(figsize=(8, 4)); L.stabilization_plot(res, fmax=8); plt.show()

In [ ]:
# YOUR CODE: look at the diagram, choose approximate column centres, then collect matching poles
# centres = [...]
# f_ssi, z_ssi = ...
raise NotImplementedError("Lab 3: pick stable poles")

In [ ]:
L.check_lab3(f_ssi, z_ssi); f_ssi, z_ssi

💬 **Q4.** Which method was more accurate, peak picking or SSI? Which gives damping?
💬 **Q5.** For Z24 you'll do this once per hour for a year (~8,000 times), then "track" each mode over time. What could go wrong? *(Modes swapping order, a mode missing in some hours... KW51 has missing frequencies for exactly these reasons.)*

## Lab 4: temperature and freezing
`L.seasonal_dataset()` gives one year of hourly mode-1 frequencies of the model bridge. The stiffness secretly follows a temperature law with a freezing term, like KW51 (and, famously, Z24, whose asphalt freezes).

**Exercise 4.** Fit your KW51 physics model
f(T) = f₀ · √(1 + a(T − 10)) · (1 + b · s·ln(1 + e^{(T_c − T)/s})), with s = 1,
using `optimize.least_squares` on **all** data. Store the estimates in `a_hat` and `Tc_hat`.

In [ ]:
d = L.seasonal_dataset()
T, f = d["T"], d["f"]
plt.figure(figsize=(5, 3.5)); plt.plot(T, f, ".", ms=1.5, alpha=.4); plt.xlabel("T [°C]"); plt.ylabel("f₁ [Hz]"); plt.show()

def physics(theta, T, s=1.0):
    f0, a, b, Tc = theta
    return f0 * np.sqrt(np.clip(1 + a * (T - 10), 1e-6, None)) * (1 + b * s * np.logaddexp(0, (Tc - T) / s))

In [ ]:
# YOUR CODE: res4 = optimize.least_squares(lambda th: physics(th, T) - f, x0=[...], bounds=(...));  a_hat, Tc_hat = ...
raise NotImplementedError("Lab 4: fit the physics law")

In [ ]:
L.check_lab4(a_hat, Tc_hat); a_hat, Tc_hat

💬 **Q6 (this is your KW51 negative result!).** Refit using **only hours with T > 3 °C** (the model never sees freezing). Can you still recover T_c? Plot both fitted curves over −5…25 °C. What does this tell you about extrapolating into an unseen regime?

## Lab 5: conformal prediction, random vs seasonal split
**Exercise 5.** Write `conformal_halfwidth(residuals, alpha)`: return the ⌈(n+1)(1−α)⌉-th smallest absolute residual. Then measure the test coverage of 90 % bands:
- (a) **Random split** with the physics model: shuffle the hours and use 50 % train, 20 % calibration, 30 % test → `cov_random`.
- (b) **Seasonal split** with the physics model: train = the first 4 months (hours 0–2,919), calibration = the next month (2,920–3,649), test = the rest → `cov_seasonal_physics`.
- (c) The same seasonal split with a **cubic polynomial** in T (`np.polyfit(T, f, 3)` / `np.polyval`) → `cov_seasonal_cubic`.

The `fit` and `coverage` helpers are provided.

In [ ]:
def fit(Ttr, ftr):
    return optimize.least_squares(lambda th: physics(th, Ttr) - ftr, x0=[np.median(ftr), -5e-4, 1e-3, 1.0],
                                  bounds=([0.5, -0.02, 0, -5], [3, 0.02, 0.1, 8])).x

def coverage(y, lo, hi):
    return np.mean((y >= lo) & (y <= hi))

In [ ]:
def conformal_halfwidth(residuals, alpha=0.1):
    # YOUR CODE
    raise NotImplementedError("Lab 5: conformal quantile")

# YOUR CODE: cov_random, cov_seasonal_physics, cov_seasonal_cubic
raise NotImplementedError("Lab 5: the three experiments")

In [ ]:
L.check_lab5(cov_random, cov_seasonal_physics, cov_seasonal_cubic); cov_random, cov_seasonal_physics, cov_seasonal_cubic

💬 **Q7.** Explain, as you would to a judge, why both models reach ~90 % on a random split, but only the physics model holds up on the seasonal split. Which temperatures did training never see? Plot both fitted curves from −5 to 25 °C.
💬 **Q7b.** In this lab the physics law is *exactly* the law that generated the data, so it extrapolates perfectly. In KW51 the physics is only approximate. What does that change? *(Hint: your real model needed a learned correction, and still failed on the cold side.)*
💬 **Q8.** How does this lab connect to your KW51 results, and what would you need to do differently for Z24?

---
**Done?**
- Write a half-page summary of the lab in your notebook.
- Then bring me your answers to Q6–Q8. I'll give feedback like a mentor.
- When the Z24 data arrive, Labs 2–3 are your toolkit for identifying its modes.